<a href="https://colab.research.google.com/github/Mbogho64/Mbogho-s-week-5-course-project/blob/main/Week_5_Assignment_%7C_Supervised_Slope_Stability_Machine_Learning_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Week 5 Assignment: Building a Supervised Machine Learning Model**

**Project Problem**

Mine and quarry slope stability depends on geotechnical properties, slope geometry, pore-water conditions and reinforcement. This project investigates whether supervised machine learning can use these parameters to predict the Factor of Safety (FS), which is an engineering indicator of slope stability. Two regression models will be trained and evaluated to determine how well they predict FS for observations not used during training.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [3]:
from google.colab import files

uploaded = files.upload()

Saving slope_stability_dataset.csv to slope_stability_dataset.csv


In [4]:
df = pd.read_csv("slope_stability_dataset.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (10000, 9)


,Unit Weight (kN/m³),Cohesion (kPa),Internal Friction Angle (°),Slope Angle (°),Slope Height (m),Pore Water Pressure Ratio,Reinforcement Type,Reinforcement Numeric,Factor of Safety (FS)
0,18.745401,21.813837,38.249958,41.907228,18.451042,0.847237,Drainage,3,2.613692
1,24.507143,19.981044,24.612800,32.964623,9.266800,0.494517,Geosynthetics,2,2.241626
2,22.319939,12.926926,28.665992,58.224926,10.686165,0.195466,Retaining Wall,0,1.568244
3,20.986585,32.327000,36.582016,20.948923,13.130201,0.736642,Drainage,3,3.000000
4,16.560186,26.448087,32.052234,39.392821,14.164400,0.418678,Soil Nailing,1,3.000000


In [5]:
print(df.columns.tolist())

['Unit Weight (kN/m³)', 'Cohesion (kPa)', 'Internal Friction Angle (°)', 'Slope Angle (°)', 'Slope Height (m)', 'Pore Water Pressure Ratio', 'Reinforcement Type', 'Reinforcement Numeric', 'Factor of Safety (FS)']


**Target and Features**

The target variable is Factor of Safety (FS), a continuous numerical measure used in slope stability assessment. The input features are unit weight, cohesion, internal friction angle, slope angle, slope height, pore water pressure ratio and reinforcement type. These parameters represent material properties, slope geometry, groundwater influence and stabilization measures that may be associated with calculated slope stability.

Reinforcement type will be treated as a categorical variable rather than as an arbitrary numerical ranking.

In [6]:
target = "Factor of Safety (FS)"

features = [
    "Unit Weight (kN/m³)",
    "Cohesion (kPa)",
    "Internal Friction Angle (°)",
    "Slope Angle (°)",
    "Slope Height (m)",
    "Pore Water Pressure Ratio",
    "Reinforcement Type"
]

X = df[features].copy()
y = df[target].copy()

print("Features:", X.shape)
print("Target:", y.shape)

Features: (10000, 7)
Target: (10000,)


**Type of Machine Learning Problem**

This project is a supervised regression problem because Factor of Safety is a continuous numerical variable. The model will learn relationships between seven input features representing geotechnical properties, slope geometry, pore-water conditions and reinforcement type, and the known Factor of Safety values. It will then use these relationships to predict Factor of Safety for observations that were not used during training.
Regression is appropriate because the objective is to estimate a numerical stability indicator rather than assign observations to predefined categories.

**Splitting the dataset into training and testing sets**

Using 80% of the observations  for training and 20% for testing and run the code as follows:

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

Training observations: 8000
Testing observations: 2000


**Preparing the data for both models**

The dataset contains numerical features and a categorical feature, Reinforcement Type. The categorical values must be converted into a format the models can process:

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "Unit Weight (kN/m³)",
    "Cohesion (kPa)",
    "Internal Friction Angle (°)",
    "Slope Angle (°)",
    "Slope Height (m)",
    "Pore Water Pressure Ratio"
]

categorical_features = ["Reinforcement Type"]

# Preprocessing for Linear Regression
preprocess_lr = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"),
     categorical_features)
])

# Preprocessing for Random Forest
preprocess_rf = ColumnTransformer([
    ("numeric", "passthrough", numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"),
     categorical_features)
])

**Selecting and training the two supervised model**

Comparing:
1. Linear regression - a baseline model that estimates a linear relationship between the inputs and Factor of Safety.

2. Random Forest Regressor - an ensemble model that can learn more complex, nonlinear relationships.

In [9]:
# Model 1: Linear Regression
linear_model = make_pipeline(
    preprocess_lr,
    LinearRegression()
)

linear_model.fit(X_train, y_train)

# Model 2: Random Forest Regression
forest_model = make_pipeline(
    preprocess_rf,
    RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
)

forest_model.fit(X_train, y_train)

print("Both models have been trained successfully.")

Both models have been trained successfully.


**Predicting Factor of Safety on the test set**

Using both models to make predictions on the test data:

In [10]:
linear_pred = linear_model.predict(X_test)
forest_pred = forest_model.predict(X_test)

predictions = pd.DataFrame({
    "Actual FS": y_test.to_numpy(),
    "Linear Regression Prediction": linear_pred,
    "Random Forest Prediction": forest_pred
})

display(predictions.head(10))

,Actual FS,Linear Regression Prediction,Random Forest Prediction
0,1.805486,1.946370,1.753022
1,3.000000,2.829880,3.000000
2,2.453911,2.237887,2.363061
3,3.000000,2.969010,3.000000
4,3.000000,3.130445,3.000000
5,0.863270,1.520726,0.896275
6,3.000000,3.705807,3.000000
7,3.000000,2.546725,2.999960
8,2.464116,2.179381,2.498111
9,2.038266,1.997504,2.179970


**Evaluating the model performance**

Using the 3 standard regression metrics learnt:

*   Mean Absolute Error(MAE) - The avearage absolute difference between actual and predicted Factor of Safety. A lower MAE is better.
*   Root Mean Squared Error(RMSE) - Measures prediction error while penalizing larger errors more strongly. A lower RMSE is better.
*   Coefficient of Determination(R²) - Measures how well predictions account for variation in the target relative to a mean-based baseline. Values that are closer to 1 generally indicate better performance.

In [11]:
def evaluate_model(name, actual, predicted):
    return {
        "Model": name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted)
    }

results = pd.DataFrame([
    evaluate_model("Linear Regression", y_test, linear_pred),
    evaluate_model("Random Forest", y_test, forest_pred)
])

display(results.round(4))

,Model,MAE,RMSE,R²
0,Linear Regression,0.2903,0.3396,0.7302
1,Random Forest,0.0224,0.0415,0.9960


**Checking overfitting, underfitting and generalization:**

The above 3 important machine learning concepts are introduced by this assignment.

To investigate them, I compared each model's performance on training data against its performance on test data as shown below:

In [12]:
comparison = []

for name, model in [
    ("Linear Regression", linear_model),
    ("Random Forest", forest_model)
]:
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    comparison.append({
        "Model": name,
        "Training RMSE": np.sqrt(
            mean_squared_error(y_train, train_pred)
        ),
        "Testing RMSE": np.sqrt(
            mean_squared_error(y_test, test_pred)
        ),
        "Training R²": r2_score(y_train, train_pred),
        "Testing R²": r2_score(y_test, test_pred)
    })

display(pd.DataFrame(comparison).round(4))

,Model,Training RMSE,Testing RMSE,Training R²,Testing R²
0,Linear Regression,0.3364,0.3396,0.7414,0.7302
1,Random Forest,0.0160,0.0415,0.9994,0.9960


**Model comparison and results discussion**

Linear Regression and Random Forest Regression were trained to predict Factor of Safety using geotechnical, geometric, pore water and reinforcement parameters. Their performance was evaluated on a separate test set using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE) and the coefficient of determination (R²).
Linear Regression achieved a test MAE of 0.2903	, RMSE of 0.3396 and R² of 0.7302. Random Forest achieved a test MAE of 0.0224, RMSE of 0.0415 and R² of 0.9960.
Based on these results, Random Forest model performed better overall because it had lower prediction errors(MAE, RMSE and higher R²).
Comparing training and test performance also provided an indication of how well each model generalizes to unseen observations.
The comparison demonstrates how different supervised learning algorithms can model relationships between geotechnical parameters and Factor of Safety. However, the results apply to the dataset used and do not, by themselves, demonstrate reliable prediction of actual mine or quarry slope failures.
